<a href="https://colab.research.google.com/github/Innovatewithapple/SingleTopicRag/blob/main/SingleTopicRag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers faiss-cpu

In [ ]:
import numpy as np
import random
import os
import torch

def setProductionSeed(isActive,seed=42):
  if isActive == True:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
      torch.cuda.manual_seed(seed)
      torch.cuda.manual_seed_all(seed)

      torch.backends.cudnn.deterministic = True
      torch.backends.cudnn.benchmark = False

setProductionSeed(True)

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from google.colab import userdata
from sentence_transformers import SentenceTransformer
import faiss
import re

In [ ]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
!kaggle datasets download -d samuelmatsuoharris/single-topic-rag-evaluation-dataset

Dataset URL: https://www.kaggle.com/datasets/samuelmatsuoharris/single-topic-rag-evaluation-dataset
License(s): MIT
single-topic-rag-evaluation-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
!unzip -q single-topic-rag-evaluation-dataset -d ./datafolder/

replace ./datafolder/documents.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [ ]:
df = pd.read_csv('/content/datafolder/documents.csv')
df.sample(5)

,index,source_url,text
0,0,https://enterthegungeon.fandom.com/wiki/Bullet...,Bullet Kin\nBullet Kin are one of the most com...
17,17,https://alanwake.fandom.com/wiki/Alan_Wake_2,Alan Wake 2\nWhy the hell did you kill Casey? ...
15,15,http://www.chakoteya.net/DoctorWho/40-1.html,Space Babies\n\nOriginal Airdate: 11 May 2024\...
1,1,https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1a...,---The Paths through the Underground/Underdark...
8,8,https://whattocook.substack.com/p/so-into-nort...,so into northern spain!\nour magical urban-plu...


In [ ]:
df.columns

Index(['index', 'source_url', 'text'], dtype='object')

In [ ]:
df['text'][0]

'Bullet Kin\nBullet Kin are one of the most common enemies. They slowly walk towards the player, occasionally firing a single bullet. They can flip tables and use them as cover. They will also deal contact damage if the player touches them.\n\nOccasionally, Bullet Kin will have assault rifles, in which case they will rapidly fire 8 bullets towards the player before reloading. When an assault rifle wielding bullet kin appears, there will often be more in the same room.\n\nOn some occasions the player will also encounter incapacitated Bullet Kin lying on the floor. These Bullet Kin are props and disintegrate upon touch. They can be found in mass quantity in Oubliette.\n\nIn the Black Powder Mine, they can also ride Minecarts. In fact, if there are any unoccupied Minecarts within the room, they will take priority by walking towards them to ride in.\n\nTrivia\nBullet Kin wield Magnums. Assault-rifle wielding Bullet Kin wield AK-47s.\nIncapacitated Bullet Kin can be found in the Oublilette 

In [ ]:
# ---- SANITY CHECK — run this before any chunking ----

# your documents
avg_doc = sum(len(row['text']) for _, row in df.iterrows()) / len(df)
max_doc = max(len(row['text']) for _, row in df.iterrows())
avg_line = sum(
    sum(len(l) for l in row['text'].split('\n') if l.strip()) /
    max(len([l for l in row['text'].split('\n') if l.strip()]), 1)
    for _, row in df.iterrows()
) / len(df)

# your model limits (you fill these from model card)
embedding_token_limit = 8192        # nomic
embedding_safe_chars  = embedding_token_limit * 4  # 1 token ≈ 4 chars

llm_token_limit = 8000              # whatever LLM you use
top_k = 3

# your chunk sizes
child_size  = 600
parent_size = 800

# calculations
child_tokens       = child_size / 4
parent_tokens      = parent_size / 4
tokens_to_llm      = top_k * parent_tokens
max_chunks_expected = max_doc / child_size

print("====== PROJECT SANITY CHECK ======")
print(f"\n--- DOCUMENTS ---")
print(f"Avg doc length:      {avg_doc:.0f} chars")
print(f"Max doc length:      {max_doc:.0f} chars")
print(f"Avg line length:     {avg_line:.0f} chars")

print(f"\n--- EMBEDDING MODEL (nomic) ---")
print(f"Token limit:         {embedding_token_limit} tokens")
print(f"Safe chunk limit:    {embedding_safe_chars} chars")
print(f"Your child_size:     {child_size} chars = ~{child_tokens:.0f} tokens")
if child_size < embedding_safe_chars:
    print(f"Status:              ✓ SAFE (child is under limit)")
else:
    print(f"Status:              ✗ DANGER (child exceeds model limit!)")

print(f"\n--- LLM CONTEXT ---")
print(f"LLM token limit:     {llm_token_limit} tokens")
print(f"Tokens sent (top_k × parent): {tokens_to_llm:.0f} tokens")
if tokens_to_llm < llm_token_limit * 0.7:  # stay under 70% of limit
    print(f"Status:              ✓ SAFE")
else:
    print(f"Status:              ✗ DANGER (too much context for LLM!)")

print(f"\n--- CHUNK COUNT ESTIMATE ---")
print(f"Max chunks expected: {max_chunks_expected:.0f} (from largest doc)")
print(f"If actual >> this:   chunking logic is broken")
print("==================================")

total_chars = sum(len(row['text']) for _, row in df.iterrows())
estimated_total_chunks = total_chars / child_size

print(f"\n--- CHUNK COUNT ESTIMATE ---")
print(f"Max chunks (largest doc): {max_chunks_expected:.0f}")
print(f"Total chunks (all docs):  {estimated_total_chunks:.0f}")  # ← add this
print(f"If actual >> this:        chunking logic is broken")

====== PROJECT SANITY CHECK ======

--- DOCUMENTS ---
Avg doc length:      35862 chars
Max doc length:      211782 chars
Avg line length:     154 chars

--- EMBEDDING MODEL (nomic) ---
Token limit:         8192 tokens
Safe chunk limit:    32768 chars
Your child_size:     600 chars = ~150 tokens
Status:              ✓ SAFE (child is under limit)

--- LLM CONTEXT ---
LLM token limit:     8000 tokens
Tokens sent (top_k × parent): 600 tokens
Status:              ✓ SAFE

--- CHUNK COUNT ESTIMATE ---
Max chunks expected: 353 (from largest doc)
If actual >> this:   chunking logic is broken

--- CHUNK COUNT ESTIMATE ---
Max chunks (largest doc): 353
Total chunks (all docs):  1195
If actual >> this:        chunking logic is broken


In [ ]:
def sentence_split(text, chunk_size=600):
    # split on single newlines (the actual structure)
    lines = [l.strip() for l in text.split('\n') if l.strip()]

    # if only one line or lines are too long, hard split
    if len(lines) == 1 or max(len(l) for l in lines) > chunk_size:
        return [text[i:i+chunk_size]for i in range(0, len(text), chunk_size)]
    chunks = []
    current = ""
    for line in lines:
        if len(current) + len(line) + 1 < chunk_size:
            current += line + " "
        else:
            if current.strip():
                chunks.append(current.strip())
            # start new chunk with overlap from previous
            current = line + " "  # start fresh, no overlap

    if current.strip():
        chunks.append(current.strip())

    return chunks

def parent_child_chunk(text,parent_size=500,child_size=150):
  # clean first
  text = re.sub(r'\n{3,}', '\n\n', text).strip()

  parents = sentence_split(text,chunk_size=parent_size)
  result = []
  for p_idx,parent in enumerate(parents):
    children = sentence_split(parent,chunk_size=child_size)
    for c_idx,child in enumerate(children):
      result.append({
          'child_text':child,
          'parent_text':parent,
          'parent_id':p_idx,
          'child_id':f'{p_idx}_{c_idx}'
      })
  return result

In [ ]:
#---------Build Chunks----------#
all_chunks = []

for idx,rows in df.iterrows():
  text = rows['text']
  source = rows['source_url']

  #Clean noise
  #text = text.replace('\n', ' ').strip()
  text = text.strip()
  text = re.sub(r'\n{3,}', '\n\n', text)  # collapse 3+ newlines into 2
  #parent-child-chunking
  chunks = parent_child_chunk(text,parent_size=800,child_size=400)

  for chunk in chunks:
    chunk['doc_index'] = idx
    chunk['source'] = source
    all_chunks.append(chunk)

print(f"Total chunks: {len(all_chunks)}")

Total chunks: 2007


In [ ]:
for i in [0, 5, 10]:
    c = all_chunks[i]
    print(f"Child  ({len(c['child_text'])} chars): {c['child_text'][:80]}")
    print(f"Parent ({len(c['parent_text'])} chars): {c['parent_text'][:80]}")
    print(f"Same? {c['child_text'] == c['parent_text']}")
    print("---")

Child  (400 chars): Bullet Kin Bullet Kin are one of the most common enemies. They slowly walk towar
Parent (666 chars): Bullet Kin Bullet Kin are one of the most common enemies. They slowly walk towar
Same? False
---
Child  (248 chars): h up with the player quickly if they attempt to take cover. They fire 4 bullets 
Parent (648 chars): Bullet Kin is also a crossover skin in the game Riverbond. Bullet Kin also has a
Same? False
---
Child  (400 chars): Trivia Although normally seen in the Abbey & Hollow, a single cardinal may be se
Parent (679 chars): Trivia Although normally seen in the Abbey & Hollow, a single cardinal may be se
Same? False
---


In [ ]:
# debug one parent
text = df['text'][0].strip()
text = re.sub(r'\n{3,}', '\n\n', text).strip()

parents = sentence_split(text, chunk_size=800)
print(f"Number of parents: {len(parents)}")
print(f"Parent 0 length: {len(parents[0])}")

children = sentence_split(parents[0], chunk_size=400)
print(f"Number of children from parent 0: {len(children)}")
for i, c in enumerate(children):
    print(f"  Child {i}: {len(c)} chars")

Number of parents: 15
Parent 0 length: 666
Number of children from parent 0: 2
  Child 0: 400 chars
  Child 1: 266 chars


In [ ]:
text = df['text'][0].strip()
text = re.sub(r'\n{3,}', '\n\n', text)

# count actual sentence endings
periods = text.count('. ')
newlines = text.count('\n')
double_newlines = text.count('\n\n')

print(f"Period+space: {periods}")
print(f"Single newlines: {newlines}")
print(f"Double newlines: {double_newlines}")
print(f"Total chars: {len(text)}")
print(f"\nFirst 500 chars:")
print(repr(text[:500]))

Period+space: 39
Single newlines: 147
Double newlines: 41
Total chars: 10654

First 500 chars:
'Bullet Kin\nBullet Kin are one of the most common enemies. They slowly walk towards the player, occasionally firing a single bullet. They can flip tables and use them as cover. They will also deal contact damage if the player touches them.\n\nOccasionally, Bullet Kin will have assault rifles, in which case they will rapidly fire 8 bullets towards the player before reloading. When an assault rifle wielding bullet kin appears, there will often be more in the same room.\n\nOn some occasions the player w'


In [ ]:
doc_0_chunks = [c for c in all_chunks if c['doc_index'] == 0]
print(f"Doc 0 chunks: {len(doc_0_chunks)}")
print(f"Expected roughly: {len(df['text'][0]) // (600-100)}")

Doc 0 chunks: 30
Expected roughly: 21


In [ ]:
# check last chunk of one doc and first chunk of next doc
for doc_idx in range(3):  # check first 3 document boundaries

    # get all chunks belonging to this doc
    doc_chunks = [c for c in all_chunks if c['doc_index'] == doc_idx]
    next_doc_chunks = [c for c in all_chunks if c['doc_index'] == doc_idx + 1]

    print(f"\n=== LAST PARENT of doc {doc_idx} ===")
    print(doc_chunks[-1]['parent_text'])

    print(f"\n=== FIRST PARENT of doc {doc_idx + 1} ===")
    print(next_doc_chunks[0]['parent_text'])

    print("\n" + "="*50)


=== LAST PARENT of doc 0 ===
Red-Caped Bullet Kin wield Magnums, but do not fire them or point them at the player. Red-Caped Bullet Kin do not deal contact damage unless they are jammed. Red-Caped Bullet Kin's design may be based on The Kid from I Wanna Be The Guy. Rooms created by the Drill can have a Red-Caped Bullet Kin spawn inside them, even if a Red-Caped Bullet Kin has already appeared on that floor. It's possible for Red-Caped Bullet Kin to appear in the Aimless Void and Secret Floors such as the Oubliette. Red-Caped Bullet Kin are not attacked by companions. Red-Caped Bullet Kin will teleport away if the room contains an enemy that cannot be killed, such as Gunreapers or Dead Blows.

=== FIRST PARENT of doc 1 ===
---The Paths through the Underground/Underdark---(9 days of travel) Wandering through the dark tunnels, the rushing sounds of the underground river begin to fade as it diverges from the cavern. You walk on for miles, the smell of hard water and wet earth. Natural cha

In [ ]:
for idx, row in df.iterrows():
    print(f"Doc {idx}: {len(row['text'])} chars")

Doc 0: 10654 chars
Doc 1: 2750 chars
Doc 2: 22187 chars
Doc 3: 20421 chars
Doc 4: 11648 chars
Doc 5: 14278 chars
Doc 6: 35485 chars
Doc 7: 17961 chars
Doc 8: 11322 chars
Doc 9: 13486 chars
Doc 10: 10203 chars
Doc 11: 62004 chars
Doc 12: 20080 chars
Doc 13: 53109 chars
Doc 14: 16592 chars
Doc 15: 31757 chars
Doc 16: 211782 chars
Doc 17: 22984 chars
Doc 18: 52550 chars
Doc 19: 75993 chars


In [ ]:
#----------Load Model------------#
model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5',trust_remote_code=True)

In [ ]:
# ---- STEP 2: EXTRACT CHILD TEXTS FOR EMBEDDING ----
# we embed children (precise retrieval)
# but store parent_text alongside for LLM context

child_texts = [c['child_text'] for c in all_chunks]
parent_texts = [c['parent_text'] for c in all_chunks]
doc_index = [c['doc_index'] for c in all_chunks]
sources = [c['source'] for c in all_chunks]

#-----------Batch Embedding-------------!
prefix = ['search_document: ' + t for t in child_texts]

embeddings = model.encode(prefix,batch_size=32,show_progress_bar=True,normalize_embeddings=True)
print(f'embedding shape: {embeddings.shape}') #(num_chunks,786)

#-----------Build faiss index-------------!
dim = embeddings.shape[1] # 786

index = faiss.IndexFlatIP(dim) # indexflatIp is dot product, but we normalized the embedding that means now its cosine similarity
index.add(embeddings.astype(np.float32))
print(f'Faiss index build, total vectors: {index.ntotal}')

In [ ]:
#------------Retrieval Function---------------!
def Retrieve(question,top_k=4):
  query_vector = model.encode(['search_query: ' + question],normalize_embeddings=True).astype(np.float32)

  #Faiss search - returns score and indices
  scores,indices = index.search(query_vector,top_k)

  results = []

  for score,idx in zip(scores[0],indices[0]):
    results.append({
        'score':float(score),
        'child_text':child_texts[idx],
        'parent_text':parent_texts[idx],
        'doc_index':doc_index[idx],
        'source':sources[idx]
    })
  return results

In [ ]:
# ---- STEP 6: TEST RETRIEVAL ----
results = Retrieve("What kind of gun does the bandana bullet kin use?", top_k=5)
for r in results:
    print(f"Score:      {r['score']:.4f}")
    print(f"Doc index:  {r['doc_index']}")
    print(f"Source:     {r['source']}")
    print(f"Child:      {r['child_text']}")
    print(f"Parent:     {r['parent_text']}")
    print("---")

Score:      0.8374
Doc index:  0
Source:     https://enterthegungeon.fandom.com/wiki/Bullet_Kin
Child:      Bandana Bullet Kin behave like regular Bullet Kin, but their fire rate is heavily increased. Bandana Bullet Kin also have a higher magazine size than Bullet Kin that wield AK-47s, making them more relentless. Trivia Bandana Bullet Kin wield Machine Pistols. Tanker Tankers behave like regular Bullet Kin, but have higher health and higher rate of fire. Tankers can be spawned by Treadnaught. Their ra
Parent:     Bandana Bullet Kin behave like regular Bullet Kin, but their fire rate is heavily increased. Bandana Bullet Kin also have a higher magazine size than Bullet Kin that wield AK-47s, making them more relentless. Trivia Bandana Bullet Kin wield Machine Pistols. Tanker Tankers behave like regular Bullet Kin, but have higher health and higher rate of fire. Tankers can be spawned by Treadnaught. Their rate of fire is slightly lower than that of Bandana Bullet Kin, but they are just

In [ ]:
qa_df = pd.read_csv('/content/single_passage_answer_questions.csv')

In [ ]:
correct = 0
missed = []

for _, row in qa_df.iterrows():
    results = Retrieve(row['question'], top_k=5)
    retrieved_docs = [r['doc_index'] for r in results]

    if row['document_index'] in retrieved_docs:
        correct += 1
    else:
        missed.append(row['question'])

total = len(qa_df)
print(f"Accuracy: {correct}/{total} = {correct/total*100:.1f}%")
print(f"\nMissed questions:")
for q in missed:
    print(f"  - {q}")

Accuracy: 37/40 = 92.5%

Missed questions:
  - What do the giants look like?
  - How do the data storage options compare?
  - What are the key topics of this article?


In [ ]:
correct_strict = 0

for _, row in qa_df.iterrows():
    results = Retrieve(row['question'], top_k=5)

    # check if answer keywords appear in any retrieved parent
    answer_words = set(row['answer'].lower().split())
    retrieved_text = " ".join([r['parent_text'].lower() for r in results])
    retrieved_words = set(retrieved_text.split())

    overlap = answer_words & retrieved_words
    if len(overlap) / len(answer_words) > 0.5:  # 50% of answer words found
        correct_strict += 1

print(f"Strict accuracy: {correct_strict}/40 = {correct_strict/40*100:.1f}%")

Strict accuracy: 31/40 = 77.5%


In [ ]:
correct = 0
missed = []

for _, row in qa_df.iterrows():
    results = Retrieve(row['question'], top_k=5)

    # combine all retrieved parent texts
    retrieved_text = " ".join([r['parent_text'].lower() for r in results])

    # check if answer is actually IN the retrieved text
    answer = row['answer'].lower()
    answer_words = answer.split()

    # check if most answer words appear in retrieved text
    found = sum(1 for w in answer_words if w in retrieved_text)
    ratio = found / len(answer_words)

    if ratio > 0.6:  # 60% of answer words found in retrieved chunks
        correct += 1
    else:
        missed.append({
            'question': row['question'],
            'answer': row['answer'],
            'ratio': ratio
        })

print(f"Real accuracy: {correct}/40 = {correct/40*100:.1f}%")
print(f"\nMissed:")
for m in missed:
    print(f"  Q: {m['question']}")
    print(f"  A: {m['answer']}")
    print(f"  Match ratio: {m['ratio']:.2f}")
    print()

Real accuracy: 32/40 = 80.0%

Missed:
  Q: What do the giants look like?
  A: One giant is burly, grey-skinned, and 20 feet tall. It wears heavy dark iron armour covered in metal torns. It is leaning against a chamber wall, carrying what looks to be two tower shields nearly as tall as itself, both bearing
menacing spikes.

The other giant  stands a tall giant in more mobile iron, clutching a dangerous-looking maul the size of Yasha, also leaning and looking disinterested.
  Match ratio: 0.43

  Q: What happens on day 2?
  A: After a few miles of winding tunnel, you emerge in a smaller grotto of stalactites and stalagmites dripping with condensation. Unsure if the same underground river, or another water source, is nearby, you can see quite a bit of ground water does funnel down into this area.

You encounter 2 ropers seeking the next burrowed entrance left by the Kryn.
  Match ratio: 0.60

  Q: What data did was used to test the prototype?
  A: Grace Hopper's Wikipedia page and Alan Tu

In [ ]:
question = "What kind of gun does the bandana bullet kin use?"
results = Retrieve(question, top_k=5)

# check if machine pistol is actually in retrieved chunks
for i, r in enumerate(results):
    has_answer = "machine pistol" in r['parent_text'].lower()
    print(f"Chunk {i}: score={r['score']:.4f} | contains answer: {has_answer}")
    print(f"  {r['parent_text'][:100]}")

Chunk 0: score=0.8374 | contains answer: True
  Bandana Bullet Kin behave like regular Bullet Kin, but their fire rate is heavily increased. Bandana
Chunk 1: score=0.7589 | contains answer: False
  In the Black Powder Mine, they can also ride Minecarts. In fact, if there are any unoccupied Minecar
Chunk 2: score=0.7533 | contains answer: True
  Bandana Bullet Kin behave like regular Bullet Kin, but their fire rate is heavily increased. Bandana
Chunk 3: score=0.7513 | contains answer: False
  Bullet Kin Bullet Kin are one of the most common enemies. They slowly walk towards the player, occas
Chunk 4: score=0.7426 | contains answer: False
  Bullet Kin is also a crossover skin in the game Riverbond. Bullet Kin also has a cameo as lower and 


# **Decoder**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# ---- STEP 1: LOAD MODEL ----
model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # ← add this
llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

In [ ]:
# ---- STEP 2: PROMPT BUILDER ----
def build_prompt(question, results):
    context = "\n---\n".join([r['parent_text'] for r in results])

    messages = [
        {"role": "system", "content": "You are a precise assistant. Answer using ONLY the context provided. If the answer is not in the context, say I don't know."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return prompt

In [ ]:
# ---- STEP 3: ANSWER FUNCTION ----
def answer(question):
    # retrieve top 5, send top 3 to LLM
    results = Retrieve(question, top_k=5)
    top_3 = results[:3]

    # build prompt
    prompt = build_prompt(question, top_3)

    # tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    input_tokens = inputs['input_ids'].shape[1]
    print(f"Input tokens: {input_tokens}")  # sanity check

    # generate
    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.1,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    # decode only the answer part
    answer_tokens = output[0][input_tokens:]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True)

In [ ]:
# ---- STEP 4: TEST ----
q1 = "What do keybullet kin drop?"
q2 = "What kind of gun does the bandana bullet kin use?"

print(f"Q: {q1}")
print(f"A: {answer(q1)}")
print("---")
print(f"Q: {q2}")
print(f"A: {answer(q2)}")

Q: What do keybullet kin drop?
Input tokens: 606
A: KeybulletKin drop a single key upon the player's death or destruction of one. In cases where they are jammed, they drop two keys instead.
---
Q: What kind of gun does the bandana bullet kin use?
Input tokens: 543
A: The Bandana Bullets use Machine Pistolas.
